# Predicting House Prices

##Problem Definition

The goal of this project is to build a Linear Regression model that predicts SalePrice using housing data from the Amazon Housing dataset. This baseline model uses Gr Liv Area as the predictor variable, and model performance will be evaluated using Root Mean Squared Percentage Error (RMSPE) .


##Data Collection

In [25]:
import statsmodels.api as sm
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

from sklearn import datasets
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn import preprocessing
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from google.colab import userdata
import os

In [3]:
url = 'https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Data/Housing.Data.csv'
url

'https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Data/Housing.Data.csv'

In [4]:
df = pd.read_csv(url)
df

,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,905101070,20,RL,62.0,14299,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,7,2007,WD,Normal,115400
1,905101330,90,RL,72.0,10791,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,Shed,500,10,2006,WD,Normal,90000
2,903454090,50,RM,50.0,9000,Pave,NaN,Reg,Bnk,AllPub,...,0,NaN,NaN,NaN,0,12,2007,WD,Normal,141000
3,533244030,60,FV,68.0,7379,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,254000
4,909252020,70,RL,60.0,7200,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,4,2009,WD,Normal,155000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2632,903231070,50,RM,52.0,6240,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,Shed,400,9,2006,WD,Normal,114500
2633,906201021,80,RL,74.0,10778,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,7,2009,WD,Normal,162000
2634,533253070,120,RL,61.0,3782,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2009,WD,Normal,211500
2635,527376100,20,RL,78.0,10140,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,8,2009,WD,Normal,165000


In [5]:
#Rows and Columns
df.shape

(2637, 81)

###Columns

####Find Identifier Columns

In [6]:
#Run code to identifier columns
identifier_cols = []

for col in df.columns:
    if df[col].nunique() == len(df):
        identifier_cols.append(col)

print(identifier_cols)

['PID']


In [7]:
#Verify all rows are unique
df['PID'].nunique()

2637

In [8]:
#Must match df['target'].nunique
len(df)

2637

####Verify Target Column doesn't have any nulls

In [9]:
#Rows with nulls in target column
df['PID'].isnull().sum()

np.int64(0)

####Feature columns

In [10]:
#Drop target column
df.drop(columns=['PID'], inplace=True)

In [11]:
#View all columns in a sorted list
# df.columns.sort_values().to_list()

In [12]:
#Rows with nulls in each
df.isnull().sum().sort_values()*1000

,0
MS SubClass,0
MS Zoning,0
Lot Area,0
Street,0
Land Contour,0
...,...
Mas Vnr Type,1607000
Fence,2109000
Alley,2457000
Misc Feature,2541000


####Numerical/Categorical columns

In [13]:
#Look for int, float, object totals at the bottom
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2637 entries, 0 to 2636
Data columns (total 80 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   MS SubClass      2637 non-null   int64  
 1   MS Zoning        2637 non-null   object 
 2   Lot Frontage     2188 non-null   float64
 3   Lot Area         2637 non-null   int64  
 4   Street           2637 non-null   object 
 5   Alley            180 non-null    object 
 6   Lot Shape        2637 non-null   object 
 7   Land Contour     2637 non-null   object 
 8   Utilities        2637 non-null   object 
 9   Lot Config       2637 non-null   object 
 10  Land Slope       2637 non-null   object 
 11  Neighborhood     2637 non-null   object 
 12  Condition 1      2637 non-null   object 
 13  Condition 2      2637 non-null   object 
 14  Bldg Type        2637 non-null   object 
 15  House Style      2637 non-null   object 
 16  Overall Qual     2637 non-null   int64  
 17  Overall Cond  

###Rows

In [14]:
df.isnull().sum().sort_values()

,0
MS SubClass,0
MS Zoning,0
Lot Area,0
Street,0
Land Contour,0
...,...
Mas Vnr Type,1607
Fence,2109
Alley,2457
Misc Feature,2541


In [15]:
#Duplicate rows
df.duplicated().sum()

np.int64(0)

In [16]:
#Missing rows
df.isnull().sum().sort_values()

,0
MS SubClass,0
MS Zoning,0
Lot Area,0
Street,0
Land Contour,0
...,...
Mas Vnr Type,1607
Fence,2109
Alley,2457
Misc Feature,2541


##Data Cleaning

In [17]:
df['SalePrice'].isna().sum()

np.int64(0)

In [18]:
df['SalePrice'].describe()

,SalePrice
count,2637.000000
mean,179986.230186
std,78309.251522
min,12789.000000
25%,129500.000000
50%,160000.000000
75%,213000.000000
max,745000.000000


In [19]:
df.dtypes.value_counts()

,count
object,43
int64,26
float64,11


Int Fields

In [20]:
df_int = df.select_dtypes(include=['int64']).drop(columns=['SalePrice'])
df_int

,MS SubClass,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,1st Flr SF,2nd Flr SF,Low Qual Fin SF,Gr Liv Area,...,Fireplaces,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold
0,20,14299,4,3,1964,1964,1005,0,0,1005,...,0,0,0,0,0,0,0,0,7,2007
1,90,10791,4,5,1967,1967,1296,0,0,1296,...,0,0,0,0,0,0,0,500,10,2006
2,50,9000,6,6,1937,1950,780,595,0,1375,...,1,0,162,0,0,126,0,0,12,2007
3,60,7379,8,5,2000,2000,975,873,0,1848,...,1,280,184,0,0,0,0,0,4,2010
4,70,7200,7,9,1936,2007,575,560,0,1135,...,0,256,0,0,0,0,0,0,4,2009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2632,50,6240,6,6,1934,1950,816,0,360,1176,...,1,112,0,0,0,0,0,400,9,2006
2633,80,10778,7,6,1990,1991,1061,0,0,1061,...,0,114,36,0,0,0,0,0,7,2009
2634,120,3782,8,5,1981,1981,1226,0,0,1226,...,2,133,78,0,0,0,0,0,9,2009
2635,20,10140,6,5,1974,1974,1350,0,0,1350,...,1,0,0,0,0,0,0,0,8,2009


In [21]:
df.select_dtypes(include=['int64']).isna().sum()*1000

,0
MS SubClass,0
Lot Area,0
Overall Qual,0
Overall Cond,0
Year Built,0
Year Remod/Add,0
1st Flr SF,0
2nd Flr SF,0
Low Qual Fin SF,0
Gr Liv Area,0


In [22]:
df['Gr Liv Area'].isna().sum()

np.int64(0)

## Push to Hugging Face

In [23]:
cols = ['SalePrice'] + df_int.columns.tolist()
df_lr = df[cols]
df_lr

,SalePrice,MS SubClass,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,1st Flr SF,2nd Flr SF,Low Qual Fin SF,...,Fireplaces,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold
0,115400,20,14299,4,3,1964,1964,1005,0,0,...,0,0,0,0,0,0,0,0,7,2007
1,90000,90,10791,4,5,1967,1967,1296,0,0,...,0,0,0,0,0,0,0,500,10,2006
2,141000,50,9000,6,6,1937,1950,780,595,0,...,1,0,162,0,0,126,0,0,12,2007
3,254000,60,7379,8,5,2000,2000,975,873,0,...,1,280,184,0,0,0,0,0,4,2010
4,155000,70,7200,7,9,1936,2007,575,560,0,...,0,256,0,0,0,0,0,0,4,2009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2632,114500,50,6240,6,6,1934,1950,816,0,360,...,1,112,0,0,0,0,0,400,9,2006
2633,162000,80,10778,7,6,1990,1991,1061,0,0,...,0,114,36,0,0,0,0,0,7,2009
2634,211500,120,3782,8,5,1981,1981,1226,0,0,...,2,133,78,0,0,0,0,0,9,2009
2635,165000,20,10140,6,5,1974,1974,1350,0,0,...,1,0,0,0,0,0,0,0,8,2009


## Convert CSV to parquet


In [31]:
parquet_file = "housing.parquet"
parquet_file

'housing.parquet'

In [32]:
df_lr.to_parquet( parquet_file, index=False)

In [33]:
!ls -l --si {parquet_file}

-rw-r--r-- 1 root root 86k Jul  1 14:29 housing.parquet


## Verify that pandas can query the local parquet file


In [34]:
df2 = pd.read_parquet( parquet_file )
df2.shape

(2637, 26)

## Authenticate to Hugging Face


In [26]:
# Retrieve the token from Colab's secret manager
os.environ["HF_TOKEN"] = userdata.get('hf_token')
_ = os.environ["HF_TOKEN"]
f"{_[:5]} ... {_[-3:]}"

'hf_rP ... BIc'

In [28]:
os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account = os.environ["HF_ACCOUNT"]
hf_account


'stephanie465337'

In [29]:
hf_repo = "Data_Science-21"
os.environ["HF_REPO"] = hf_repo
hf_repo


'Data_Science-21'

In [30]:
# Log in automatically without an interactive prompt
!hf auth login --token $HF_TOKEN


Hint: A new version of huggingface_hub (1.21.0) is available! You are using version 1.20.1.
To update, run: hf update
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Colab` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Push to Hugging Face


In [35]:
%%capture hf_upload
%%bash

hf upload \
  --type dataset \
  ${HF_ACCOUNT}/${HF_REPO} \
  housing.parquet

In [36]:
print(hf_upload.stdout)

✓ Uploaded
  url: https://huggingface.co/datasets/Stephanie465337/Data_Science-21/commit/379fa0f4f5fc833d6baf5faef9fb18e7f3fd2eae



## Read parquet from Hugging Face


In [37]:
# Construct the base Hugging Face URL for your dataset files
hf_url = f"https://huggingface.co/datasets/{hf_account}/{hf_repo}/resolve/main/housing.parquet"
hf_url

'https://huggingface.co/datasets/stephanie465337/Data_Science-21/resolve/main/housing.parquet'

In [38]:
df3 = pd.read_parquet( hf_url )
df3.shape

(2637, 26)

In [39]:
df3.iloc[:,:5].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2637 entries, 0 to 2636
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   SalePrice     2637 non-null   int64
 1   MS SubClass   2637 non-null   int64
 2   Lot Area      2637 non-null   int64
 3   Overall Qual  2637 non-null   int64
 4   Overall Cond  2637 non-null   int64
dtypes: int64(5)
memory usage: 103.1 KB
